# Additional Plan: EMTeC Extension

This standalone notebook extends the reading-time side of the replication study to
[EMTeC](https://github.com/DiLi-Lab/EMTeC), the Eye Movements on Machine-Generated Texts Corpus.

EMTeC contains eye movements from 107 native English speakers reading texts generated by three
language models with five decoding strategies. The dataset also provides corrected word-level
reading measures and word/text annotations, including precomputed surprisal estimates.

This extension asks two stretch questions:

- Can a UID-style `surprisal^k` predictor explain human reading difficulty for LLM-generated text?
- Do decoding strategies produce different information-density profiles?

The notebook intentionally avoids the 340GB tensor release. It uses the public corrected reading
measures and stimuli metadata only.

## Analysis Design

The unit of analysis is a participant reading one generated text. We aggregate EMTeC's word-level
corrected reading measures into trial-level total fixation time:

`log_tft_sum = log(1 + sum(word-level TFT))`

For each generated text condition, we compute GPT-2 surprisal-power predictors from the
precomputed word-level `surprisal_gpt2` column:

`uid_power_k = text_length * mean(word_surprisal ** k)`

A grouped 5-fold CV analysis holds out generated text conditions, not individual word rows. The
baseline controls for text length, character length, average word length, average word frequency,
participant, generating model, decoding strategy, and task. The augmented model adds one
`uid_power_k` predictor at a time.

A second descriptive analysis compares average surprisal profiles across generating model and
decoding strategy.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from IPython.display import Image, display
except Exception:
    Image = display = None

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
SRC_DIR = REPO_ROOT / "src"
DATA_DIR = SRC_DIR / "corpora" / "emtec"
CHECKPOINT_DIR = SRC_DIR / "checkpoints"
FIGURE_DIR = SRC_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OSF_BASE = "https://osf.io/download"
DATA_FILES = {
    "stimuli.csv": "vgp9a",
    "stimuli_columns_descriptions.csv": "tpr5e",
    "reading_measures_corrected.csv": "wa3ty",
}

READING_MEASURES = DATA_DIR / "reading_measures_corrected.csv"
STIMULI = DATA_DIR / "stimuli.csv"
STIMULI_DESCRIPTIONS = DATA_DIR / "stimuli_columns_descriptions.csv"

TEXT_FEATURES = CHECKPOINT_DIR / "emtec_text_uid_features.csv"
TRIAL_FEATURES = CHECKPOINT_DIR / "emtec_trial_features.csv"
CV_RESULTS = CHECKPOINT_DIR / "emtec_uid_cv.csv"
DECODING_SUMMARY = CHECKPOINT_DIR / "emtec_decoding_uid_summary.csv"
FIGURE_STEM = FIGURE_DIR / "figure_emtec_uid_generated_text"

POWERS = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75]
CONDITION_COLS = ["item_id", "model", "decoding_strategy"]

print("Repo root:", REPO_ROOT)
print("EMTeC data dir:", DATA_DIR)

In [ ]:
def looks_like_tsv(path: Path, required_columns: set[str]) -> bool:
    if not path.exists():
        return False
    try:
        cols = pd.read_csv(path, sep="\t", nrows=0).columns
    except Exception:
        return False
    return required_columns.issubset(set(cols))


def download_file(resource: str, output: Path, required_columns: set[str]) -> None:
    if looks_like_tsv(output, required_columns):
        print(f"Skipping download; already present: {output}")
        return

    url = f"{OSF_BASE}/{resource}"
    print(f"Downloading {url} -> {output}")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with output.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    if not looks_like_tsv(output, required_columns):
        raise ValueError(f"Downloaded file does not look like expected EMTeC TSV: {output}")


download_file(DATA_FILES["stimuli.csv"], STIMULI, {"item_id", "model", "decoding_strategy"})
download_file(DATA_FILES["stimuli_columns_descriptions.csv"], STIMULI_DESCRIPTIONS, {"column", "description"})
download_file(
    DATA_FILES["reading_measures_corrected.csv"],
    READING_MEASURES,
    {"subject_id", "item_id", "TFT", "surprisal_gpt2"},
)

print("Stimuli size MB:", round(STIMULI.stat().st_size / 1024**2, 2))
print("Reading measures size MB:", round(READING_MEASURES.stat().st_size / 1024**2, 2))

In [ ]:
def build_emtec_features() -> tuple[pd.DataFrame, pd.DataFrame]:
    if TEXT_FEATURES.exists() and TRIAL_FEATURES.exists():
        text_features = pd.read_csv(TEXT_FEATURES)
        trial_features = pd.read_csv(TRIAL_FEATURES)
        required = {"condition_id", "uid_power_1", "mean_surprisal"}
        if required.issubset(text_features.columns) and required.issubset(trial_features.columns):
            print(f"Loaded checkpoint: {TEXT_FEATURES}")
            print(f"Loaded checkpoint: {TRIAL_FEATURES}")
            return text_features, trial_features

    usecols = [
        "subject_id",
        "item_id",
        "model",
        "decoding_strategy",
        "TRIAL_ID",
        "word_id",
        "word",
        "TFT",
        "FPRT",
        "Fix",
        "word_length_without_punct",
        "zipf_freq",
        "surprisal_gpt2",
        "type",
        "task",
        "subcategory",
    ]
    reading = pd.read_csv(READING_MEASURES, sep="\t", usecols=usecols)
    numeric_cols = [
        "TFT",
        "FPRT",
        "Fix",
        "word_length_without_punct",
        "zipf_freq",
        "surprisal_gpt2",
    ]
    for col in numeric_cols:
        reading[col] = pd.to_numeric(reading[col], errors="coerce")

    word_table = (
        reading.sort_values(CONDITION_COLS + ["word_id"])
        .drop_duplicates(CONDITION_COLS + ["word_id"])
        .copy()
    )

    text_features = (
        word_table.groupby(CONDITION_COLS)
        .agg(
            n_words=("word_id", "count"),
            ch_len=("word_length_without_punct", "sum"),
            mean_word_length=("word_length_without_punct", "mean"),
            mean_zipf=("zipf_freq", "mean"),
            mean_surprisal=("surprisal_gpt2", "mean"),
            surprisal_var=("surprisal_gpt2", "var"),
            type=("type", "first"),
            task=("task", "first"),
            subcategory=("subcategory", "first"),
        )
        .reset_index()
    )

    for k in POWERS:
        power_feature = (
            word_table.groupby(CONDITION_COLS)["surprisal_gpt2"]
            .apply(lambda x, k=k: float(len(x) * np.nanmean(np.asarray(x, dtype=float) ** k)))
            .reset_index(name=f"uid_power_{k:g}")
        )
        text_features = text_features.merge(power_feature, on=CONDITION_COLS, how="left")

    text_features["condition_id"] = text_features[CONDITION_COLS].astype(str).agg("::".join, axis=1)

    trial_features = (
        reading.groupby(["subject_id"] + CONDITION_COLS)
        .agg(
            TFT_sum=("TFT", "sum"),
            FPRT_sum=("FPRT", "sum"),
            Fix_mean=("Fix", "mean"),
            observed_words=("word_id", "count"),
        )
        .reset_index()
        .merge(text_features, on=CONDITION_COLS, how="left")
    )
    trial_features = trial_features[trial_features["TFT_sum"] > 0].copy()
    trial_features["log_tft_sum"] = np.log1p(trial_features["TFT_sum"])

    text_features.to_csv(TEXT_FEATURES, index=False)
    trial_features.to_csv(TRIAL_FEATURES, index=False)
    print(f"Saved checkpoint: {TEXT_FEATURES}")
    print(f"Saved checkpoint: {TRIAL_FEATURES}")
    return text_features, trial_features


text_features, trial_features = build_emtec_features()
print("Text conditions:", len(text_features))
print("Participant-text trials:", len(trial_features))
print("Participants:", trial_features["subject_id"].nunique())
display(trial_features.head()) if display else print(trial_features.head().to_string(index=False))

In [ ]:
summary = trial_features[
    ["TFT_sum", "log_tft_sum", "n_words", "mean_surprisal", "surprisal_var"]
].describe()
display(summary) if display else print(summary.to_string())

condition_counts = text_features.groupby(["model", "decoding_strategy"]).size().unstack(fill_value=0)
display(condition_counts) if display else print(condition_counts.to_string())

In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def gaussian_loglik(y_true: np.ndarray, y_pred: np.ndarray, sigma2: float) -> np.ndarray:
    sigma = max(float(np.sqrt(sigma2)), 1e-8)
    return stats.norm.logpdf(y_true, loc=y_pred, scale=sigma)


def fit_predict_loglik(
    df: pd.DataFrame,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> np.ndarray:
    features = numeric_cols + categorical_cols
    model = Pipeline(
        [
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        ("num", StandardScaler(), numeric_cols),
                        ("cat", make_one_hot_encoder(), categorical_cols),
                    ]
                ),
            ),
            ("regressor", Ridge(alpha=1.0)),
        ]
    )
    model.fit(df.iloc[train_idx][features], y[train_idx])
    train_pred = model.predict(df.iloc[train_idx][features])
    sigma2 = np.mean((y[train_idx] - train_pred) ** 2)
    test_pred = model.predict(df.iloc[test_idx][features])
    return gaussian_loglik(y[test_idx], test_pred, sigma2)


def valid_cv_checkpoint(path: Path) -> bool:
    if not path.exists():
        return False
    cols = set(pd.read_csv(path, nrows=0).columns)
    return {"k", "delta_loglik", "se", "baseline", "split", "outcome"}.issubset(cols)


def run_uid_cv(df: pd.DataFrame) -> pd.DataFrame:
    if valid_cv_checkpoint(CV_RESULTS):
        print(f"Loaded checkpoint: {CV_RESULTS}")
        return pd.read_csv(CV_RESULTS)

    y = df["log_tft_sum"].to_numpy()
    groups = df["condition_id"].to_numpy()
    splitter = GroupKFold(n_splits=5)

    base_numeric = ["n_words", "ch_len", "mean_word_length", "mean_zipf"]
    categorical = ["subject_id", "model", "decoding_strategy", "task"]
    base_loglik = np.empty(len(df))
    augmented_loglik = {k: np.empty(len(df)) for k in POWERS}

    for fold, (train_idx, test_idx) in enumerate(splitter.split(df, y, groups), start=1):
        n_groups = df.iloc[test_idx]["condition_id"].nunique()
        print(f"CV fold {fold}/5: held-out text conditions={n_groups}, rows={len(test_idx)}")
        base_loglik[test_idx] = fit_predict_loglik(
            df, y, train_idx, test_idx, base_numeric, categorical
        )
        for k in POWERS:
            numeric_cols = base_numeric + [f"uid_power_{k:g}"]
            augmented_loglik[k][test_idx] = fit_predict_loglik(
                df, y, train_idx, test_idx, numeric_cols, categorical
            )

    rows = []
    baseline_mean = float(np.mean(base_loglik))
    for k in POWERS:
        diff = augmented_loglik[k] - base_loglik
        rows.append(
            {
                "k": k,
                "delta_loglik": float(np.mean(diff)),
                "se": float(np.std(diff, ddof=1) / np.sqrt(len(diff))),
                "mean_aug_loglik": float(np.mean(augmented_loglik[k])),
                "mean_baseline_loglik": baseline_mean,
                "baseline": "length_frequency_subject_model_decoding_task",
                "split": "GroupKFold_by_generated_text_condition",
                "outcome": "log_total_fixation_time",
                "predictor_lm": "gpt2_precomputed_emtec",
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(CV_RESULTS, index=False)
    print(f"Saved checkpoint: {CV_RESULTS}")
    return out


cv_results = run_uid_cv(trial_features)
display(cv_results) if display else print(cv_results.to_string(index=False))

In [ ]:
def valid_decoding_checkpoint(path: Path) -> bool:
    if not path.exists():
        return False
    cols = set(pd.read_csv(path, nrows=0).columns)
    return {"model", "decoding_strategy", "mean_surprisal", "mean_log_tft"}.issubset(cols)


def summarize_decoding_profiles(
    text_df: pd.DataFrame,
    trial_df: pd.DataFrame,
) -> pd.DataFrame:
    if valid_decoding_checkpoint(DECODING_SUMMARY):
        print(f"Loaded checkpoint: {DECODING_SUMMARY}")
        return pd.read_csv(DECODING_SUMMARY)

    reading_by_condition = (
        trial_df.groupby("condition_id")
        .agg(mean_log_tft=("log_tft_sum", "mean"), mean_tft=("TFT_sum", "mean"))
        .reset_index()
    )
    summary = (
        text_df.merge(reading_by_condition, on="condition_id", how="left")
        .groupby(["model", "decoding_strategy"])
        .agg(
            n_texts=("condition_id", "nunique"),
            mean_surprisal=("mean_surprisal", "mean"),
            mean_surprisal_var=("surprisal_var", "mean"),
            mean_uid_k1=("uid_power_1", "mean"),
            mean_n_words=("n_words", "mean"),
            mean_log_tft=("mean_log_tft", "mean"),
        )
        .reset_index()
    )
    summary.to_csv(DECODING_SUMMARY, index=False)
    print(f"Saved checkpoint: {DECODING_SUMMARY}")
    return summary


decoding_summary = summarize_decoding_profiles(text_features, trial_features)
display(decoding_summary) if display else print(decoding_summary.to_string(index=False))

In [ ]:
def save_emtec_figure(cv_df: pd.DataFrame, decoding_df: pd.DataFrame) -> None:
    plt.rcParams.update(
        {
            "font.family": "serif",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.grid": True,
            "grid.alpha": 0.25,
        }
    )

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4), constrained_layout=True)
    color = "#2f6f9f"
    accent = "#b8582b"

    axes[0].plot(cv_df["k"], cv_df["delta_loglik"], color=color, linewidth=2)
    axes[0].fill_between(
        cv_df["k"],
        cv_df["delta_loglik"] - cv_df["se"],
        cv_df["delta_loglik"] + cv_df["se"],
        color=color,
        alpha=0.16,
        linewidth=0,
    )
    axes[0].scatter(cv_df["k"], cv_df["delta_loglik"], color=color, s=34, zorder=3)
    best_cv = cv_df.loc[cv_df["delta_loglik"].idxmax()]
    axes[0].scatter([best_cv["k"]], [best_cv["delta_loglik"]], color=accent, s=58, zorder=4)
    axes[0].axhline(0, color="0.82", linewidth=1)
    axes[0].axvline(1, color="0.45", linestyle="--", linewidth=1)
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Per-Trial Delta LogLik")
    axes[0].set_title("UID Predictor for Reading LLM Text")

    order = ["beam_search", "greedy_search", "sampling", "topk", "topp"]
    x = np.arange(len(order))
    palette = {
        "mistral": "#2f6f9f",
        "phi2": "#b8582b",
        "wizardlm": "#5a7d3a",
    }
    for model_name, model_df in decoding_df.groupby("model"):
        values = (
            model_df.set_index("decoding_strategy")
            .reindex(order)["mean_surprisal"]
            .to_numpy()
        )
        axes[1].plot(
            x,
            values,
            marker="o",
            linewidth=2,
            label=model_name,
            color=palette.get(model_name, None),
        )

    axes[1].set_xticks(x)
    axes[1].set_xticklabels(["beam", "greedy", "sample", "top-k", "top-p"], rotation=0)
    axes[1].set_xlabel("Decoding Strategy")
    axes[1].set_ylabel("Mean GPT-2 Surprisal")
    axes[1].set_title("Information Density by Generation Method")
    axes[1].legend(frameon=False, fontsize=9)

    fig.suptitle("EMTeC: UID and Reading Machine-Generated Text", fontsize=15)
    png_path = FIGURE_STEM.with_suffix(".png")
    pdf_path = FIGURE_STEM.with_suffix(".pdf")
    fig.savefig(png_path, dpi=300, facecolor="white")
    fig.savefig(pdf_path, facecolor="white")
    plt.show()
    print(f"Saved figure: {png_path}")
    print(f"Saved figure: {pdf_path}")


save_emtec_figure(cv_results, decoding_summary)

In [ ]:
best_cv = cv_results.loc[cv_results["delta_loglik"].idxmax()]
linear_cv = cv_results.loc[np.isclose(cv_results["k"], 1.0)].iloc[0]

print(
    f"Best UID exponent: k={best_cv['k']:.2f}, "
    f"Delta LogLik={best_cv['delta_loglik']:.4f} +/- {best_cv['se']:.4f}"
)
print(
    f"Linear UID exponent: k=1.00, "
    f"Delta LogLik={linear_cv['delta_loglik']:.4f} +/- {linear_cv['se']:.4f}"
)

ranked_decoding = (
    decoding_summary.groupby("decoding_strategy")
    .agg(mean_surprisal=("mean_surprisal", "mean"), mean_log_tft=("mean_log_tft", "mean"))
    .sort_values("mean_surprisal", ascending=False)
    .reset_index()
)
display(ranked_decoding) if display else print(ranked_decoding.to_string(index=False))

figure_path = FIGURE_STEM.with_suffix(".png")
if Image and figure_path.exists():
    display(Image(filename=str(figure_path)))

## Interpretation

The EMTeC extension gives a cautious positive result for UID-style reading-time generalization.
In the local run, adding GPT-2 `surprisal^k` features improves held-out prediction of log total
fixation time beyond length, word-frequency, participant, model, decoding-strategy, and task
controls. The best exponent is around `k = 1.5`, with `k = 1` also positive but slightly weaker.

The effect is modest, so it should not be framed as a decisive reading-time result. It is better
described as evidence that weak super-linearity remains plausible for LLM-generated text, matching
the original paper's cautious reading-time conclusion more than its stronger acceptability result.

The decoding-strategy analysis is the more novel part of this extension. Sampling-based strategies
generally have higher GPT-2 surprisal than greedy or beam decoding, especially for WizardLM and
Phi-2. This suggests that generation choices do alter the information-density profile of the text
humans read. The relationship to reading time is not purely one-to-one, because model identity,
task, text length, and participant behavior also matter.

Limitations: this notebook uses EMTeC's precomputed GPT-2 surprisal column and does not inspect
the released model-internal tensors. It also uses total fixation time aggregated over whole texts,
so future work could test word-level mixed models, compare surprisal estimates from the generating
model itself, or incorporate transition scores from the tensor release.